코드 lstm-basic

In [ ]:
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes,
                 num_layers=2, bidirectional=True, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        # 양방향이면 hidden_dim*2, 단방향이면 hidden_dim
        d = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Linear(d, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        # x: (batch, seq_len), lengths: (batch,) 실제 길이
        emb = self.embedding(x)
        # 가변 길이 시퀀스 패킹 (패딩 무시)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        # h_n: (num_layers * num_directions, batch, hidden_dim)
        # 마지막 층의 양방향 은닉을 결합
        if self.lstm.bidirectional:
            h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_final = h_n[-1]
        return self.fc(self.dropout(h_final))

코드 sentiment-lstm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 모델 (앞서 정의한 LSTMClassifier 사용)
model = LSTMClassifier(
    vocab_size=20000, embedding_dim=128,
    hidden_dim=128, num_classes=3,
    num_layers=2, bidirectional=True
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    for x_batch, y_batch, lengths in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch, lengths)
        loss = criterion(logits, y_batch)
        loss.backward()
        # 그래디언트 클리핑 (RNN 학습 안정화)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

코드 timeseries-prep

In [ ]:
import pandas as pd
import numpy as np

# 원본 데이터 로드
df = pd.read_csv('korean_news_2020_2024.csv', parse_dates=['date'])
print(f"전체 기사 수: {len(df)}")
print(f"기간: {df['date'].min()} ~ {df['date'].max()}")

# 일별·토픽별 기사 수 집계 (피벗)
daily = df.pivot_table(
    index='date',
    columns='topic',
    aggfunc='size',
    fill_value=0
)

# 누락 날짜 채우기 (전체 기간 연속화)
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
daily = daily.reindex(full_range, fill_value=0)
daily.index.name = 'date'

# 7일 이동 평균 (단기 잡음 평활화)
daily_smoothed = daily.rolling(window=7, center=True, min_periods=1).mean()

print(f"시계열 행: {daily.shape[0]}, 열(토픽): {daily.shape[1]}")
print(daily.head())

코드 timeseries-lstm

In [ ]:
import torch
import torch.nn as nn

class TopicTimeSeriesLSTM(nn.Module):
    def __init__(self, num_topics=7, hidden_dim=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_topics,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=0.3,
            batch_first=True
        )
        # 양방향이므로 hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, num_topics)

    def forward(self, x):
        # x: (batch, seq_len=30, num_topics=7)
        out, _ = self.lstm(x)
        # 마지막 시점의 출력만 사용 (다음 날 예측)
        last = out[:, -1, :]
        return self.fc(last)

# 학습 흐름 (요약)
model = TopicTimeSeriesLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 시계열 분할 (시간 순서 유지!)
# 학습: 2020~2023, 검증: 2024 전반, 시험: 2024 후반
for epoch in range(50):
    model.train()
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()